# `tax` — profile, choose columns & trim

Trim type: **column**. Choose **10 columns**, **80,000 rows**. Save `tax_10c_80000r.csv`.

> Output columns: `c0`…`c9`.

In [3]:
import os
import numpy as np
import pandas as pd

NAME       = "tax"
N_ROWS     = 1000000
N_COLS     = 10
TRIM_ROWS  = "sample"   # head | sample

RAW_PATH   = "Tax_r1000001_c15.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ","
OUT_DELIM  = DELIM
HAS_HEADER = True
ENCODING   = "utf-8"
ON_BAD_LINES = None

## 1. View the raw data

In [4]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (1000000, 15)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14
0,Chengqi,Piera,M,567,1000000,BROWNSVILLE,OH,43721,M,N,11500,0.927767,0,2600,0
1,Yasmina,Marsala,M,402,1000000,MARSLAND,NE,69354,M,N,36500,6.840000,0,206,0
2,Jesas,Verykios,F,863,1000000,CANDLER,FL,32111,S,Y,38500,0.000000,0,0,0
3,Hiroyuji,Rahmat,M,765,1000000,HILLSDALE,IN,47854,S,Y,19000,3.400000,1000,0,1000
4,Mordechai,Leifert,M,580,1000000,KINTA,OK,74552,S,Y,40500,6.250000,1000,0,1000


In [5]:
raw.dtypes

c0      object
c1      object
c2      object
c3       int64
c4       int64
c5      object
c6      object
c7       int64
c8      object
c9      object
c10      int64
c11    float64
c12      int64
c13      int64
c14      int64
dtype: object

## 2. Profile: cardinality, top-value %, and group skew

In [6]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,10000,0,0.00,1.00,0.01,100.00,145,1.45
c1,10000,0,0.00,1.00,0.01,100.00,142,1.42
c2,2,0,0.00,0.00,50.22,500000.00,502171,1.00
c3,273,0,0.00,0.03,2.21,3663.00,22113,6.04
c4,22113,0,0.00,2.21,0.03,45.22,273,6.04
c5,18728,0,0.00,1.87,1.66,53.40,16563,310.19
c6,52,0,0.00,0.01,2.30,19230.77,22979,1.19
c7,41826,0,0.00,4.18,0.03,23.91,329,13.76
c8,2,0,0.00,0.00,50.12,500000.00,501186,1.00


## 3. Choose columns (cardinality mix + id/super-key)

In [7]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c2', 'c3', 'c4', 'c5', 'c7', 'c8', 'c11', 'c12', 'c13', 'c14']


## 4. Trim to the chosen columns x exact rows

In [8]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (1000000, 10)


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9
0,M,571,1002728,HAMPTON,23661,M,5.750000,0,1800,0
1,F,605,1001484,JAVA,57452,M,0.000000,0,0,0
2,F,860,1005248,CANTON,6019,M,5.000000,0,24500,0
3,F,336,1001162,JEFFERSON,28640,M,7.777972,0,6600,0
4,M,503,1000375,MITCHELL,97750,S,9.000000,159,0,159


## 5. Save the trimmed CSV

In [9]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote tax_10c_1000000r.csv (1000000, 10)
reloaded: (1000000, 10)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


## 6. Check selected cardinality and skew

In [10]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

output shape : (1000000, 10)
columns      : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9
0,M,571,1002728,HAMPTON,23661,M,5.750000,0,1800,0
1,F,605,1001484,JAVA,57452,M,0.000000,0,0,0
2,F,860,1005248,CANTON,6019,M,5.000000,0,24500,0
3,F,336,1001162,JEFFERSON,28640,M,7.777972,0,6600,0
4,M,503,1000375,MITCHELL,97750,S,9.000000,159,0,159


In [11]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,2,0,0.00,0.00,50.22,500000.00,502171,1.00
c1,273,0,0.00,0.03,2.21,3663.00,22113,6.04
c2,22113,0,0.00,2.21,0.03,45.22,273,6.04
c3,18728,0,0.00,1.87,1.66,53.40,16563,310.19
c4,41826,0,0.00,4.18,0.03,23.91,329,13.76
c5,2,0,0.00,0.00,50.12,500000.00,501186,1.00
c6,2483,87,0.01,0.25,21.72,402.58,217189,539.50
c7,28,0,0.00,0.00,63.58,35714.29,635821,17.80
c8,28,0,0.00,0.00,63.78,35714.29,637776,17.86
